# Above limit

In [ ]:
import os
import json
import pickle
from datetime import datetime, timedelta
import networkx as nx
import pandas as pd
import numpy as np
from collections import defaultdict
from multiprocessing import Pool
import matplotlib.pyplot as plt
from tqdm import tqdm

from PyPDF2 import PdfMerger


In [ ]:
plt.rcParams["font.size"] = 24

ipv_color = {4: "tab:blue", 6: "tab:green"}


In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

working_dir = parameters["WORKING_DIR"]
data_dir = parameters["DATA_DIR"]
data_raw_dir = parameters["DATA_RAW_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
collectors = parameters["COLLECTORS"]
image_dir = parameters["IMAGE_DIR"]
tier1 = parameters["TIER1"]
tier1_asns = [item["asn"] for item in tier1]


## Load data

In [ ]:
start_date_obj = datetime.strptime(start_date, "%Y-%m-%d")
end_date_obj = datetime.strptime(end_date, "%Y-%m-%d")

delta = timedelta(hours=8)
all_times = []
current = start_date_obj
while current < end_date_obj:
    all_times.append(current)
    current += delta


### PeeringDB data

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_peeringdb = pd.read_pickle(filename)
df_peeringdb.head(2)


In [ ]:
filename = f"{data_dir}/processed/timeseries_prefix_announced_visibility.pkl"

with open(filename, "rb") as fd:
    announced_prefixes = pickle.load(fd)


### Number of Peers

In [ ]:
filename = f"{data_dir}/processed/peers/df_peers_ipv4.pkl"
with open(filename, "rb") as f:
    peers_ipv4 = pickle.load(f)

filename = f"{data_dir}/processed/peers/df_peers_ipv6.pkl"
with open(filename, "rb") as f:
    peers_ipv6 = pickle.load(f)


### Selected ASNs

In [ ]:
fd = open(f"{data_dir}/processed/selected_asns.pkl", "rb")
selected_asns = pickle.load(fd)
fd.close()

len(selected_asns)


### Filter Prefix Origin, Announced and PeeringDB data to selected ASNs

In [ ]:
announced_prefixes = {
    asn: announced_prefixes[asn] for asn in selected_asns if asn in announced_prefixes
}

df_peeringdb = df_peeringdb[df_peeringdb["asn"].isin(selected_asns)].copy()


## Find excedence events

In [ ]:
import matplotlib.dates as mdates

def plot_excedence_event(event, lines=['prefixes', 'limit', 'peers'], high_visibility="visibility_95"):

    asn = event["asn"]
    ipv = event["ipv"]
    excedence_date = event["date"]
    start_date = event["start_date"]
    end_date = event["end_date"]
    min_peers = event["min_peers"]
    max_peers = event["max_peers"]
    min_prefixes = event["min_prefixes"] if "min_prefixes" in event else None
    max_prefixes = event["max_prefixes"] if "max_prefixes" in event else None

    announced_prefixes_asn_ipv = announced_prefixes[asn][ipv][high_visibility]
    announced_prefixes_asn_ipv = {
        k: v for k, v in sorted(announced_prefixes_asn_ipv.items(), key=lambda x: x[0])
    }

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers_ipv_asn_dates = peers_ipv_asn["datetime"]
    peers_ipv_asn_num_peers = peers_ipv_asn["num_peers"]
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in zip(peers_ipv_asn_dates, peers_ipv_asn_num_peers)
    }
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in sorted(peers_ipv_asn.items(), key=lambda x: x[0])
    }

    # by default asn should exist in peeringdb since we are using selected_asns
    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    # however, it may be that there is no limit for this ASN and IP version, so we need to check if the limit is set for this ASN and IP version
    prefix_limits_asn_ipv_date = prefix_limits_asn["dates"]
    prefix_limits_asn_ipv_count = prefix_limits_asn[f"limits_ipv{ipv}"]

    prefix_limits_asn_ipv = {
        date: count
        for date, count in zip(prefix_limits_asn_ipv_date, prefix_limits_asn_ipv_count)
    }

    plt.figure(figsize=(16, 6))

    # plt.title(f"ASN {asn} - IPv{ipv}")

    plt.ylabel("Number of Prefixes / Limit")

    # shade the sub-window that the companion update-level zoom figure expands, so the RIB
    # panel (left) and the 5-min zoom (right) read as one pair in the 4x2 grid.
    if event.get("zoom_start") and event.get("zoom_end"):
        plt.axvspan(event["zoom_start"], event["zoom_end"], color="tab:red", alpha=0.10, zorder=0)

    if "prefixes" in lines:
        plt.step(
            announced_prefixes_asn_ipv.keys(),
            announced_prefixes_asn_ipv.values(),
            color="tab:green",
            lw=3,
            where="mid",
            marker="o",
        )
        text_note = start_date + timedelta(hours=24)
        plt.text(
            text_note,
            announced_prefixes_asn_ipv[text_note] * 1.1,
            announced_prefixes_asn_ipv[text_note],
            color="tab:green",
            ha="center",
            va="bottom",
            alpha=0.8,
            fontweight="bold",
        )


    # --- Prefix limit ---
    if "limit" in lines:
        plt.step(
            prefix_limits_asn_ipv.keys(),
            prefix_limits_asn_ipv.values(),
            where="mid",
            label="PeeringDB Prefix Limit",
            marker="o",
            lw=3,
            color="tab:orange",
        )

        text_note = start_date + timedelta(hours=24 * 2)
        plt.text(
            text_note,
            prefix_limits_asn_ipv[text_note] * 1.05,
            prefix_limits_asn_ipv[text_note],
            color="tab:orange",
            ha="center",
            va="bottom",
            alpha=0.8,
            fontweight="bold",
        )

        # --- Excedence event ---
        plt.scatter(
            x=excedence_date,
            y=announced_prefixes_asn_ipv[excedence_date],
            color="tab:red",
            marker="o",
            s=150,
            zorder=5,
        )

    plt.xlim(
        start_date,
        end_date,
    )
    plt.xticks(rotation=45, ha="center")

    min_y = 0
    max_y = (
        max(
            max(announced_prefixes_asn_ipv.values()),
            max(prefix_limits_asn_ipv.values()),
        )
        * 1.1
    )

    if min_prefixes is not None and max_prefixes is not None:
        min_y = min_prefixes
        max_y = max_prefixes

    plt.ylim(
        min_y,
        max_y,
    )

    # enforce the y tick format to be integer since we are dealing with number of prefixes and peers
    plt.gca().yaxis.set_major_locator(plt.MaxNLocator(integer=True))

    ax1 = plt.gca()   # prefix axis -- draw x-gridlines here (they don't render on the twin)
    # --- Change the y-axis to the right for the number of peers ---
    plt.twinx()

    plt.ylabel("Number of Peers")

    if "peers" in lines:
        # --- Number of peers ---
        plt.step(
            peers_ipv_asn.keys(),
            peers_ipv_asn.values(),
            where="mid",
            marker="o",
            lw=3,
            color="tab:purple",
        )
        text_note = end_date - timedelta(hours=24)
        plt.text(
            text_note,
            peers_ipv_asn[text_note] * 0.9,
            peers_ipv_asn[text_note],
            color="tab:purple",
            ha="center",
            va="bottom",
            alpha=0.8,
            fontweight="bold",
        )

    plt.xlim(
        start_date,
        end_date,
    )

    plt.ylim(min_peers, max_peers)

    ax = plt.gca()
    # ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)               # horizontal y gridlines
    if event.get("peer_yticks"):
        ax.set_yticks(event["peer_yticks"])    # share the right axis with the update panel

    # faint 8h vertical gridlines (00/08/16 UTC), same faint look as the y grid; drawn on the
    # prefix axis because minor gridlines silently do not render on a twinx() axis.
    # ax1.set_axisbelow(True)
    # for _g in pd.date_range(start_date, end_date, freq="1d"):
    #     ax1.axvline(_g, color="grey", lw=0.6, alpha=0.30, zorder=0)
    
    if len(lines) != 3 :
        suffix = "_".join(lines)
        filename = f'{image_dir}/case_study/{asn}_ipv{ipv}_{excedence_date.strftime("%Y_%m_%d_%H")}_{suffix}.pdf'    
    else:
        filename = f'{image_dir}/case_study/{asn}_ipv{ipv}_{excedence_date.strftime("%Y_%m_%d_%H")}.pdf'

    plt.savefig(filename, bbox_inches="tight", dpi=300)
    plt.savefig(filename.replace(".pdf", ".png"), bbox_inches="tight", dpi=300)

    plt.show()

In [ ]:
# zoom_start/zoom_end shade the sub-window that the companion 5-min update zoom (nb 11.8)
# expands; keep these identical to ZOOM in 11.8 so the RIB | zoom pair lines up in the grid.
case_study_events = [
    {
        "asn": 44901,
        "ipv": 6,
        "date": datetime(2025, 1, 15, 16, 0, 0),
        "start_date": datetime(2025, 1, 11, 0, 0, 0),
        "end_date": datetime(2025, 1, 20, 0, 0, 0),
        "min_peers": 450,
        "max_peers": 950,
        "peer_yticks": [500, 600, 700, 800, 900],
        "zoom_start": datetime(2025, 1, 15, 4, 0, 0),
        "zoom_end": datetime(2025, 1, 16, 12, 0, 0),
    },
    {
        "asn": 25273,
        "ipv": 4,
        "date": datetime(2025, 9, 5, 8, 0, 0),
        "start_date": datetime(2025, 9, 1),
        "end_date": datetime(2025, 9, 10),
        "min_peers": 10,
        "max_peers": 20,
        "zoom_start": datetime(2025, 9, 4, 19, 0, 0),
        "zoom_end": datetime(2025, 9, 5, 15, 0, 0),
    },
    {
        "asn": 52920,
        "ipv": 4,
        "date": datetime(2025, 8, 12),
        "start_date": datetime(2025, 8, 8),
        "end_date": datetime(2025, 8, 16),
        "min_peers": 0,
        "max_peers": 30,
        "zoom_start": datetime(2025, 8, 11, 6, 0, 0),
        "zoom_end": datetime(2025, 8, 12, 2, 0, 0),
    },
    {
        "asn": 52603,
        "ipv": 6,
        "date": datetime(2025, 9, 2),
        "start_date": datetime(2025, 8, 28),
        "end_date": datetime(2025, 9, 6),
        "min_peers": 0,
        "max_peers": 25,
        "min_prefixes": 0,
        "max_prefixes": 15,
        "zoom_start": datetime(2025, 9, 1, 8, 0, 0),
        "zoom_end": datetime(2025, 9, 2, 4, 0, 0),
    },
]


for event in case_study_events:
    plot_excedence_event(
        event,
    )
    plot_excedence_event(
        event,
        lines=['prefixes'],
    )
    plot_excedence_event(
        event,
        lines=['prefixes', 'limit'],
    )
    # break

In [ ]:
plt.figure(figsize=(15, 2.25))
plt.axis("off")
plt.step([], [], color="tab:green", lw=8, label="Announced Prefixes")
plt.step([], [], color="tab:orange", lw=8, label="Max-Prefix Limit")
plt.step([], [], color="tab:purple", lw=8, label="Number of Peers")
plt.scatter([], [], color="tab:red", lw=8, label="Exceedance Event", s=100, marker="o")

plt.legend(loc="upper right", ncol=2, frameon=False, fontsize=30)
plt.savefig(f"{image_dir}/case_study/legend.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/case_study/legend.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Unified single-row legend for the 4x2 case-study grid: the RIB panels (11.5) contribute the
# exceedance dot + shaded zoom window; the update zooms (11.8) are plain step lines (no markers,
# no RIB-detection line as of the 2026-08 review).
handles = [
    Line2D([], [], color="tab:green", lw=8, label="Announced Prefixes"),
    Line2D([], [], color="tab:orange", lw=8, label="Max-Prefix Limit"),
    Line2D([], [], color="tab:purple", lw=8, label="Number of Peers"),
    Line2D([], [], color="tab:red", lw=0, marker="o", markersize=12, label="Exceedance Event"),
    Patch(facecolor="tab:red", alpha=0.10, label="Zoom Window"),
]

plt.figure(figsize=(18, 1))
plt.axis("off")
plt.legend(handles=handles, loc="upper center", ncol=5, frameon=False, fontsize=20)
plt.savefig(f"{image_dir}/case_study/legend_single_row.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/case_study/legend_single_row.png", bbox_inches="tight", dpi=300)
plt.show()


## Tier-1 peer loss across impactful events

In [ ]:
with open(f"{data_dir}/processed/critical_excedence_events.json", "r") as f:
    critical_events = json.load(f)

print(f"IPv4 events: {len(critical_events['4'])}")
print(f"IPv6 events: {len(critical_events['6'])}")


In [ ]:
for ipv in ["4", "6"]:
    events = critical_events[ipv]

    events_with_tier1 = [
        e
        for e in events
        if any(peer[1] == "Tier-1" for peer in e.get("lost_major_peers", []))
    ]

    # Collect unique Tier-1 ASNs lost across all such events
    tier1_asns_lost = set()
    for e in events_with_tier1:
        for peer in e["lost_major_peers"]:
            if peer[1] == "Tier-1":
                tier1_asns_lost.add(peer[0])

    print(f"IPv{ipv}:")
    print(f"  Total impactful events:              {len(events)}")
    print(f"  Events with >= 1 Tier-1 lost:        {len(events_with_tier1)}")
    print(f"  Unique Tier-1 ASNs lost:             {sorted(tier1_asns_lost)}")
    print()


In [ ]:
SIGNIFICANT_TIERS = {"Tier-1", "Major"}

for ipv in ["4", "6"]:
    events = critical_events[ipv]

    events_with_tier1 = [
        e
        for e in events
        if any(peer[1] == "Tier-1" for peer in e.get("lost_major_peers", []))
    ]
    events_with_significant = [
        e
        for e in events
        if any(peer[1] in SIGNIFICANT_TIERS for peer in e.get("lost_major_peers", []))
    ]

    print(f"IPv{ipv}:")
    print(f"  Total impactful events:              {len(events)}")
    print(f"  Events with >= 1 Tier-1 lost:        {len(events_with_tier1)}")
    print(f"  Events with >= 1 Tier-1 or Major:    {len(events_with_significant)}")
    print()
